# 📖 Notebook 3: Right to Erasure (GDPR Article 17)

**Goal**: Implement the "right to be forgotten" — the most technically challenging GDPR requirement — with cascading deletes across both paired regions.

## Learning Objectives

By the end of this notebook, you'll understand:
- What GDPR Article 17 requires (right to erasure)
- Why deletion is harder than it sounds in distributed systems
- How to cascade deletes across related tables
- How to handle deletion across paired regions
- How to maintain audit trails even after data is deleted

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 08-enterprise/gdpr-paired-regions
docker compose up -d
```

### Visualization

- **Adminer**: http://localhost:8081  
  Watch users disappear from both region databases as we process erasure requests.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
from datetime import datetime

EU_WEST_CONFIG = {
    "host": "localhost",
    "port": 55433,
    "database": "gdpr_eu_west",
    "user": "demo",
    "password": "demo"
}

EU_NORTH_CONFIG = {
    "host": "localhost",
    "port": 55434,
    "database": "gdpr_eu_north",
    "user": "demo",
    "password": "demo"
}

def get_connection(region):
    config = EU_WEST_CONFIG if region == "eu-west" else EU_NORTH_CONFIG
    return psycopg2.connect(**config)

# Verify setup
for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    print(f"✅ {region}: {cur.fetchone()[0]} users")
    conn.close()

## 1. What is the Right to Erasure?

**GDPR Article 17** gives a data subject the right to obtain erasure of their personal data **on one of the grounds listed in Article 17(1)** — the data is no longer necessary, consent is withdrawn and there is no other lawful basis, they object under Article 21 and no overriding legitimate grounds exist, the processing was unlawful, and so on.

Note the shape of that: it is **not** an unconditional "delete me" button. It is a right that attaches to specific circumstances. In practice most consumer services grant it anyway, because arguing is more expensive than deleting.

What you must do:
1. Respond **without undue delay and in any event within one month** of receipt (Article 12(3)). That month can be **extended by two further months** for complex or numerous requests — but you must tell the data subject within the first month, with reasons. The lab uses 30 days as a working SLA; the legal clock is "one month", which is not the same thing in February.
2. Erase from **all systems** — databases, replicas, backups, caches, logs, search indexes, analytics, data warehouses.
3. Article 19: **notify each recipient** to whom the data was disclosed, unless that proves impossible or involves disproportionate effort — and tell the data subject who those recipients were if they ask.
4. Article 17(2): if you made the data **public**, take reasonable steps to inform other controllers who are processing it.

### When Can You Refuse?

Article 17(3) lists the exemptions. Erasure does not apply to the extent processing is necessary for:
- **Compliance with a legal obligation** under Union or member-state law (this is where statutory accounting retention lands — the period is set by *national* law, not GDPR: roughly 6 years in Ireland, 7 in the Netherlands, 10 for invoices in Germany and France. "7 years everywhere" is a myth.)
- **Archiving in the public interest, scientific or historical research, or statistics** (Article 89 safeguards apply)
- **Establishment, exercise or defence of legal claims** (e.g. an active dispute)
- **Freedom of expression and information**; **public interest in public health**; **exercise of official authority**

Crucially the exemption is scoped *to the extent necessary*. A tax obligation over an invoice does not license you to keep the person's phone number and marketing preferences.

### Why Is This Hard?

```
User asks: "Delete all my data"

You need to find and delete:
├── users table          → PII (name, email, phone)
├── addresses table      → PII (street, city)
├── orders table         → linked to user, contains purchase history
├── consent_log table    → paradox: proves consent, but contains PII
├── EU-West database     → primary copy
├── EU-North database    → replica copy       ← different local ids!
├── backups              → nightly snapshots, PITR window, long-term retention
├── analytics systems    → user behavior data
└── third-party services → email provider, payment processor, etc.
```

Three of those lines are where real systems fail, and this notebook drills into
each:

1. **The replica has different primary keys.** `SERIAL` sequences are per-database.
   Deleting `WHERE id = 2` in both regions works only by coincidence.
2. **Backups cannot be edited.** You cannot `DELETE` a row out of a snapshot that
   was taken last Tuesday, and if you could, you would have destroyed the backup's
   integrity. There is a standard answer and it is not "delete from backups".
3. **"We anonymised it" is usually false.** Most anonymisation is pseudonymisation,
   and pseudonymised data is still personal data with the full erasure right
   attached to it.

Let's implement this step by step.

In [ ]:
# ── First, let's see all the data we have for a specific user ──

def show_user_data_footprint(user_id, region):
    """Shows ALL data we have for a user across all tables."""
    conn = get_connection(region)
    cur = conn.cursor()

    print(f"\n📋 Data Footprint for User {user_id} in {region.upper()}")
    print("=" * 60)

    # Users table
    cur.execute("SELECT * FROM users WHERE id = %s", (user_id,))
    user = cur.fetchone()
    if user:
        print(f"\n👤 users table:")
        print(f"   Name: {user[2]}, Email: {user[1]}, Phone: {user[3]}")
        print(f"   DOB: {user[4]}, Country: {user[5]}, Region: {user[6]}")
    else:
        print("\n👤 users table: (no record)")

    # Addresses
    cur.execute("SELECT * FROM addresses WHERE user_id = %s", (user_id,))
    addresses = cur.fetchall()
    print(f"\n🏠 addresses table: {len(addresses)} record(s)")
    for addr in addresses:
        print(f"   {addr[2]}, {addr[3]} {addr[4]}, {addr[5]}")

    # Orders
    cur.execute("SELECT * FROM orders WHERE user_id = %s", (user_id,))
    orders = cur.fetchall()
    print(f"\n🛒 orders table: {len(orders)} record(s)")
    for order in orders:
        print(f"   {order[3]} — €{order[4]} ({order[6]})")

    # Consent log
    cur.execute("SELECT * FROM consent_log WHERE user_id = %s", (user_id,))
    consents = cur.fetchall()
    print(f"\n📜 consent_log table: {len(consents)} record(s)")
    for c in consents:
        print(f"   {c[2]}: {c[3]} (from IP {c[4]})")

    conn.close()


# Show data for Anna de Vries. Look her up by email rather than hard-coding an
# id: `users.id` is a per-database SERIAL and this notebook is about to spend a
# whole section on why that distinction matters.
def local_id_for(email, region):
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT id FROM users WHERE email = %s", (email,))
    row = cur.fetchone()
    conn.close()
    return row[0] if row else None

anna_id = local_id_for("anna.devries@example.nl", "eu-west")
print(f"Anna de Vries is local id {anna_id} in eu-west (it may differ in eu-north).")
show_user_data_footprint(anna_id, "eu-west")

## 2. The Erasure Request Workflow

A proper GDPR erasure process looks like this:

```
User: "Delete my data"
       │
       ▼
  ┌─────────────┐
  │ Create       │ ← Log the request (BEFORE deleting anything)
  │ erasure      │
  │ request      │
  └──────┬──────┘
         │
         ▼
  ┌─────────────┐
  │ Check legal  │ ← Can we legally delete? (tax records, lawsuits)
  │ holds        │
  └──────┬──────┘
         │
    ┌────┴────┐
    ▼         ▼
  DELETE    DENY
  from all  with
  regions   reason
    │
    ▼
  ┌─────────────┐
  │ Confirm to   │
  │ user         │
  └─────────────┘
```

In [ ]:
# ── Erasure Request System ─────────────────────────────────

# Tables that contain PII linked to a user.
# Order matters — delete from child tables first to respect
# foreign key constraints (even though we use ON DELETE CASCADE,
# explicit ordering is a best practice for auditability).
PII_TABLES = [
    "consent_log",   # child of users
    "orders",        # child of users
    "addresses",     # child of users
    "users",         # parent table — delete last
]


def submit_erasure_request(user_id, user_email, reason, region):
    """
    Step 1: Log the erasure request before deleting anything.
    GDPR requires proof that you received and processed the request.
    """
    conn = get_connection(region)
    cur = conn.cursor()

    cur.execute("""
        INSERT INTO erasure_requests (user_id, user_email, reason, status)
        VALUES (%s, %s, %s, 'pending')
        RETURNING id
    """, (user_id, user_email, reason))

    request_id = cur.fetchone()[0]
    conn.commit()
    conn.close()

    print(f"📥 Erasure request #{request_id} created for user {user_id} ({user_email})")
    print(f"   Reason: {reason}")
    print(f"   Status: pending")
    return request_id


def check_legal_holds(user_id, region):
    """
    Step 2: Check if there are legal reasons we CANNOT delete this data.
    
    Common holds:
    - Statutory accounting retention (Article 17(3)(b): "compliance with a
      legal obligation"). The period comes from NATIONAL law, not GDPR, and
      it varies: ~6 years IE, 7 years NL, 10 years for invoices DE and FR.
      We use 7 years as a stand-in; a real implementation needs the period
      for the member state whose law applies to that transaction.
    - Active or anticipated legal claims (Article 17(3)(e))
    - Regulatory investigations

    The exemption is scoped "to the extent necessary". It covers the invoice,
    not the customer's marketing preferences — which is why the erasure below
    de-links the order rather than keeping the whole user record.
    """
    conn = get_connection(region)
    cur = conn.cursor()

    # Check for recent orders (simulate tax retention requirement)
    cur.execute("""
        SELECT COUNT(*) FROM orders
        WHERE user_id = %s
        AND created_at > NOW() - INTERVAL '7 years'
        AND status = 'completed'
    """, (user_id,))
    recent_orders = cur.fetchone()[0]
    conn.close()

    holds = []
    if recent_orders > 0:
        holds.append({
            "type": "tax_retention",
            "reason": f"User has {recent_orders} order(s) within the assumed 7-year statutory retention period.",
            "action": "Orders will be DE-LINKED from the user (user_id set to NULL), not deleted."
        })

    return holds


# Demo: Submit an erasure request for a specific user
print("📮 Submitting Erasure Request")
print("=" * 50)

# Let's use Anna de Vries as our example — resolved by email, per region.
ANNA_EMAIL = "anna.devries@example.nl"
anna_west_id = local_id_for(ANNA_EMAIL, "eu-west")

request_id = submit_erasure_request(
    user_id=anna_west_id,
    user_email=ANNA_EMAIL,
    reason="User requested account deletion via privacy settings page",
    region="eu-west"
)

# Check legal holds
print("\n⚖️  Checking Legal Holds...")
holds = check_legal_holds(anna_west_id, "eu-west")
if holds:
    for hold in holds:
        print(f"   ⚠️  Hold: {hold['type']}")
        print(f"      {hold['reason']}")
        print(f"      Action: {hold['action']}")
else:
    print("   ✅ No legal holds — full deletion allowed")

# Anna has seeded orders from 2024, so this branch must fire. If the seed data
# ever changes and she has no orders, the notebook would quietly demonstrate
# the trivial path and the whole legal-hold lesson would vanish.
assert anna_west_id is not None, (
    f"{ANNA_EMAIL} is not present in eu-west — start from fresh containers "
    f"(docker compose down -v && docker compose up -d) before running this notebook"
)
assert holds, (
    "expected a tax-retention hold for user 1 — the seed data should give her "
    "completed orders inside the retention window, otherwise this notebook "
    "never exercises the interesting branch"
)

## 🚫 Bad → ✅ Best: Naive DELETE Leaves Orphans

Before building the proper erasure function, let's see why a naive one-line
`DELETE FROM users WHERE id = ?` is **not** GDPR-compliant.

### ❌ Bad: "just delete the user row"

If a schema has a child table **without** `ON DELETE CASCADE` (very common in
legacy systems), deleting only the parent row leaves "orphan" rows behind:

```
users (deleted)          addresses (still there!)
┌────┐                   ┌────┬─────────┬──────────────┐
│ id │                   │ id │ user_id │ street       │
├────┤                   ├────┼─────────┼──────────────┤
│ 99 │ ← deleted         │ 42 │   99    │ Privacy Lane │
└────┘                   └────┴─────────┴──────────────┘
                           ↑ orphan PII — GDPR violation!
```

The orphan rows still contain PII (a street address), but they now point at a
user that doesn't exist. The data is **still there**, just harder to find —
which is exactly what auditors look for.

### ✅ Best: cascade + explicit per-table deletes + audit log

That is what the `execute_erasure()` function below does:
1. Explicitly deletes from **every** child table (don't trust CASCADE alone).
2. Keeps a row in `erasure_requests` as proof the request was fulfilled.
3. Writes a `data_residency_log` entry for each table touched.


In [ ]:
# ── Demo: the naive DELETE anti-pattern ─────────────────────
# We create a throw-away table WITHOUT cascade to prove the point.

conn = get_connection("eu-west")
cur = conn.cursor()

# Child table with NO foreign key, NO cascade — legacy-style
cur.execute("""
    CREATE TABLE IF NOT EXISTS legacy_addresses (
        id       SERIAL PRIMARY KEY,
        user_id  INTEGER,   -- no FK, no cascade (legacy schema)
        street   VARCHAR(255)
    )
""")

# Insert a demo user + a legacy-address row for them
cur.execute("""
    INSERT INTO users (email, full_name, country_code, home_region, consent_given)
    VALUES ('naive.demo@example.de', 'Naive Demo', 'DE', 'eu-west', TRUE)
    ON CONFLICT (email) DO UPDATE SET full_name = EXCLUDED.full_name
    RETURNING id
""")
demo_id = cur.fetchone()[0]
cur.execute("INSERT INTO legacy_addresses (user_id, street) VALUES (%s, %s)",
            (demo_id, "Privacy Lane 1"))
conn.commit()

# ❌ Naive deletion — only the parent row
cur.execute("DELETE FROM users WHERE id = %s", (demo_id,))
conn.commit()

# What remains?
cur.execute("SELECT COUNT(*) FROM users WHERE id = %s", (demo_id,))
users_left = cur.fetchone()[0]
cur.execute("SELECT COUNT(*), MAX(street) FROM legacy_addresses WHERE user_id = %s", (demo_id,))
orphans, street = cur.fetchone()

print("❌ After naive DELETE FROM users:")
print(f"   users rows remaining      : {users_left}  (good — user is gone)")
print(f"   legacy_addresses orphans  : {orphans}      (BAD — PII still there!)")
print(f"   leaked PII                : street = {street!r}")

# The anti-pattern must actually leak, or this cell teaches nothing.
assert users_left == 0, "the naive DELETE should at least remove the parent row"
assert orphans == 1 and street == "Privacy Lane 1", (
    f"expected the naive DELETE to strand {street!r} in legacy_addresses; got "
    f"orphans={orphans}. If a CASCADE was added to the demo table this stopped "
    f"reproducing the failure it exists to show."
)

# ✅ The fix: delete child rows too (or use ON DELETE CASCADE from the start)
cur.execute("DELETE FROM legacy_addresses WHERE user_id = %s", (demo_id,))
cur.execute("DROP TABLE legacy_addresses")   # cleanup demo table
conn.commit(); conn.close()

print("\n✅ Lesson: always enumerate *every* table with PII before erasing a user.")
print("   The execute_erasure() function below does exactly that.")


In [ ]:
# ── The Core Erasure Function ──────────────────────────────

def resolve_local_user_id(user_email, region):
    """
    Find the row id this person has *in this particular database*.

    (Same lookup as the local_id_for() shim above; this is the documented
    version, and the one the rest of the notebook uses.)

    This function is the whole point of the section. `users.id` is a SERIAL,
    and the two regions have completely independent sequences: the same person
    can be id 7 in eu-west and id 19 in eu-north. An earlier version of this
    notebook ran `DELETE ... WHERE id = %s` with the SAME id against both
    regions. It appeared to work only because docker-compose seeds both
    containers from the same init.sql, so the ids happened to line up. The
    moment anything is written to one region and replicated to the other, they
    diverge — and then a cross-region erasure deletes SOMEBODY ELSE in the
    replica while leaving the requester's data intact. That is two GDPR
    incidents from one line of code.

    Cross-region operations must key on a stable subject identifier. Here that
    is the email (UNIQUE in our schema). In a real system you would carry an
    application-assigned subject id (a UUID) that is generated once and
    replicated verbatim, precisely so you never have to trust a local sequence.
    """
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT id FROM users WHERE email = %s", (user_email,))
    row = cur.fetchone()
    conn.close()
    return row[0] if row else None


def execute_erasure(user_email, request_id, region):
    """
    Step 3: Actually delete (or de-link) this person's data in ONE region.

    Keyed on user_email — see resolve_local_user_id() for why this must not be
    a caller-supplied numeric id.

    This handles:
    - Deleting from every PII table, in child-before-parent order
    - De-linking data that must be retained (statutory retention on orders)
    - Logging every deletion for audit
    """
    user_id = resolve_local_user_id(user_email, region)
    if user_id is None:
        # "Nothing to erase here" is still a completed outcome FOR THIS REGION,
        # and it has to be recorded as one. Leaving the request stuck at
        # 'processing' would make the completeness audit in Notebook 4 report a
        # permanently unfinished erasure for a region that is genuinely clean.
        print(f"\n   ℹ️  {user_email} has no record in {region.upper()} — nothing to erase here")
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("""
            UPDATE erasure_requests
            SET status = 'completed', completed_at = NOW(), completed_by = 'system'
            WHERE id = %s
        """, (request_id,))
        conn.commit(); conn.close()
        return []

    conn = get_connection(region)
    cur = conn.cursor()
    deletion_report = []

    try:
        print(f"\n🗑️  Executing erasure for {user_email} in {region.upper()} (local id={user_id})")
        print("-" * 50)

        # Check legal holds for this region
        holds = check_legal_holds(user_id, region)
        has_tax_hold = any(h["type"] == "tax_retention" for h in holds)

        # Step 3a: Delete consent logs (no legal requirement to keep)
        cur.execute("DELETE FROM consent_log WHERE user_id = %s", (user_id,))
        count = cur.rowcount
        deletion_report.append(("consent_log", count, "deleted"))
        print(f"   ✅ consent_log: {count} records deleted")

        # Step 3b: Handle orders (may need anonymization for tax)
        if has_tax_hold:
            # Anonymize instead of delete — keep financial data, remove PII
            cur.execute("""
                UPDATE orders
                SET user_id = NULL
                WHERE user_id = %s
            """, (user_id,))
            count = cur.rowcount
            deletion_report.append(("orders", count, "de-linked"))
            print(f"   ⚠️  orders: {count} records DE-LINKED (statutory retention hold)")
            print( "       (user_id set to NULL — the invoice survives, the link does not.")
            print( "        Whether the remainder is truly anonymous depends on what else")
            print( "        the row carries. See the pseudonymisation section below.)")
        else:
            cur.execute("DELETE FROM orders WHERE user_id = %s", (user_id,))
            count = cur.rowcount
            deletion_report.append(("orders", count, "deleted"))
            print(f"   ✅ orders: {count} records deleted")

        # Step 3c: Delete addresses
        cur.execute("DELETE FROM addresses WHERE user_id = %s", (user_id,))
        count = cur.rowcount
        deletion_report.append(("addresses", count, "deleted"))
        print(f"   ✅ addresses: {count} records deleted")

        # Step 3d: Delete the user record itself
        cur.execute("DELETE FROM users WHERE id = %s", (user_id,))
        count = cur.rowcount
        deletion_report.append(("users", count, "deleted"))
        print(f"   ✅ users: {count} records deleted")

        # Step 3e: Log the deletion in the residency log
        for table, count, action in deletion_report:
            if count > 0:
                cur.execute("""
                    INSERT INTO data_residency_log
                        (user_id, action, source_region, table_name, record_id, reason)
                    VALUES (%s, 'delete', %s, %s, %s,
                            'GDPR Article 17 erasure request #' || %s::text)
                """, (user_id, region, table, user_id, request_id))

        # Step 3f: Update the erasure request status
        cur.execute("""
            UPDATE erasure_requests
            SET status = 'completed', completed_at = NOW(), completed_by = 'system'
            WHERE id = %s
        """, (request_id,))

        conn.commit()
        print(f"\n   ✅ Erasure complete in {region}")
        return deletion_report

    except Exception as e:
        conn.rollback()
        print(f"   ❌ Erasure failed: {e}")
        raise
    finally:
        conn.close()


# Execute erasure in the PRIMARY region only — deliberately.
# We want to see what a half-finished erasure looks like before we fix it.
ANNA = "anna.devries@example.nl"

print("🗑️  Processing Erasure Request (primary region only, on purpose)")
print("=" * 50)
report = execute_erasure(user_email=ANNA, request_id=request_id, region="eu-west")

assert report, "erasure returned an empty report — did the user exist in eu-west?"
assert any(t == "users" and c == 1 for t, c, _ in report), \
    f"the users row was not deleted in eu-west; report={report}"

In [ ]:
# ── Verify: Is the data really gone? ───────────────────────

print("🔍 Verification: Is Anna de Vries' data deleted?")
print("=" * 50)

anna_west = resolve_local_user_id(ANNA, "eu-west")
print(f"\n📦 EU-WEST (where we ran the erasure): local id = {anna_west}")
if anna_west is None:
    print("   ✅ users row is gone")

# ── Now the question almost nobody asks in the demo ──────────────
anna_north = resolve_local_user_id(ANNA, "eu-north")
print(f"\n📦 EU-NORTH (the paired replica): local id = {anna_north}")
if anna_north is not None:
    show_user_data_footprint(anna_north, "eu-north")

print("\n" + "!" * 62)
print("🚨 THE ERASURE IS NOT DONE.")
print("   We deleted from the primary. The paired region — which exists")
print("   precisely so that a copy survives when the primary is destroyed —")
print("   still holds her name, email, phone, date of birth and address.")
print("   Every property that makes a replica good for disaster recovery")
print("   makes it a place erasure has to reach.")
print("!" * 62)

assert anna_west is None, "she should be gone from the region we erased"
assert anna_north is not None, (
    "the replica should STILL hold her at this point — that gap is the lesson "
    "of this section. If she is already gone from eu-north, something erased "
    "her early and the demo below proves nothing."
)

# Also check that the erasure request record still exists
# (this is the audit trail — it stays even after deletion)
conn = get_connection("eu-west")
cur = conn.cursor()
cur.execute("""
    SELECT id, user_email, reason, status, requested_at, completed_at
    FROM erasure_requests
    WHERE user_email = 'anna.devries@example.nl'
""")
request = cur.fetchone()
conn.close()

print("\n📋 Erasure Request Audit Trail (preserved):")
print(f"   Request #{request[0]}: {request[1]}")
print(f"   Reason: {request[2]}")
print(f"   Status: {request[3]}")
print(f"   Requested: {request[4]}")
print(f"   Completed: {request[5]}")
print("\n💡 The erasure request record itself is NOT deleted.")
print("   It is your evidence that you received and acted on the request —")
print("   Article 5(2) accountability. Note its status already says 'completed',")
print("   which right now is a LIE: the replica still has her.")
print()
print("⚠️  And note what this evidence row costs you: erasure_requests stores")
print("   her plaintext email address, indefinitely, in both regions. An email")
print("   address is personal data. You have erased a person from your system")
print("   by writing their identifier into a table you promised never to purge.")
print("   The usual resolutions:")
print("     • store a keyed hash (HMAC) of the identifier, not the identifier;")
print("     • or keep the plaintext under a short, documented retention period")
print("       with a legitimate-interests assessment behind it.")
print("   Either way it is a decision to make on purpose, not a leftover.")

## 3. Cross-Region Erasure

We just watched a "completed" erasure leave the person intact in the replica.
Fixing it is conceptually easy — run the erasure in every region — with one trap
that is very easy to fall into.

### ⚠️ The trap: local ids are not the same person

`users.id` is a `SERIAL`. Each database has its own sequence. Anna might be id 1
in eu-west and id 1 in eu-north today only because both containers loaded the
same `init.sql`. As soon as either region takes a write the other did not, the
sequences drift — and after that, `DELETE FROM users WHERE id = 7` run against
both regions deletes **two different people**.

The failure mode is nasty because it is silent and it is symmetric: the
requester's data survives in one region, and an innocent third party is erased
in the other. Both are Article 17 incidents. Neither raises an exception.

Cross-region operations must key on something that means the same thing
everywhere: a natural key like the (UNIQUE) email, or better, an
application-generated subject UUID that is replicated as data. Never a local
sequence value.

```
Erasure request received
        │
        ▼
  ┌─────────────┐    ┌─────────────┐
  │ Delete from  │    │ Delete from  │
  │ EU-West      │    │ EU-North     │
  │ (primary)    │    │ (replica)    │
  └──────┬──────┘    └──────┬──────┘
         │                   │
         └────────┬──────────┘
                  ▼
         Confirm to user:
         "All copies deleted"
```

In [ ]:
# ── Proof that local ids drift between regions ─────────────────────
# We write one person into eu-west and the SAME person into eu-north, the way a
# real system would when a signup happens on each side of the pair. Then we look
# at what id each database gave them.

DRIFT_EMAIL = "drift.demo@example.nl"

for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("DELETE FROM users WHERE email = %s", (DRIFT_EMAIL,))
    cur.execute("""
        INSERT INTO users (email, full_name, country_code, home_region, consent_given)
        VALUES (%s, 'Drift Demo', 'NL', 'eu-west', TRUE)
    """, (DRIFT_EMAIL,))
    conn.commit()
    conn.close()

west_id = resolve_local_user_id(DRIFT_EMAIL, "eu-west")
north_id = resolve_local_user_id(DRIFT_EMAIL, "eu-north")

print("🔢 SAME PERSON, TWO DATABASES")
print("=" * 50)
print(f"   eu-west  local id : {west_id}")
print(f"   eu-north local id : {north_id}")

if west_id != north_id:
    print(f"\n   ❌ The ids differ. `DELETE WHERE id = {west_id}` in eu-north would")
    print(f"      hit whoever happens to be id {west_id} there — not this person.")
    # Show exactly who would have been wrongly erased.
    conn = get_connection("eu-north")
    cur = conn.cursor()
    cur.execute("SELECT email, full_name FROM users WHERE id = %s", (west_id,))
    victim = cur.fetchone()
    conn.close()
    if victim:
        print(f"      In this database that is: {victim[1]} <{victim[0]}>")
        print( "      Erasing them would be an unlawful deletion of a third party's")
        print( "      data, on top of failing to erase the person who asked.")
else:
    print("\n   ⚠️  They happen to match right now (both sequences are at the same")
    print("      point). That is luck, not a guarantee — and code that depends on")
    print("      luck is code that fails in production, not in the demo.")

assert resolve_local_user_id(DRIFT_EMAIL, "eu-west") is not None
assert resolve_local_user_id(DRIFT_EMAIL, "eu-north") is not None

# Clean up — by email, naturally.
for region in ["eu-west", "eu-north"]:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("DELETE FROM users WHERE email = %s", (DRIFT_EMAIL,))
    conn.commit()
    conn.close()
print("\n🧹 Drift demo cleaned up.")


In [ ]:
# ── Full Cross-Region Erasure ──────────────────────────────

ALL_REGIONS = ["eu-west", "eu-north"]

def verify_erased_everywhere(user_email, regions=ALL_REGIONS):
    """
    Independently re-query every region for any trace of this person.
    Returns {region: [table names that still hold a row]}.

    This is separate from the erasure function on purpose. A deletion routine
    that reports its own success is a deletion routine that reports success when
    it silently did nothing — the erasure_requests row in the previous section
    said 'completed' while the replica still held everything.
    """
    residue = {}
    for region in regions:
        local_id = resolve_local_user_id(user_email, region)
        conn = get_connection(region)
        cur = conn.cursor()
        found = []
        if local_id is not None:
            found.append("users")
            for table in ("addresses", "orders", "consent_log"):
                cur.execute(f"SELECT COUNT(*) FROM {table} WHERE user_id = %s", (local_id,))
                if cur.fetchone()[0]:
                    found.append(table)
        conn.close()
        if found:
            residue[region] = found
    return residue


def full_gdpr_erasure(user_email, reason):
    """
    Complete GDPR Article 17 erasure across ALL paired regions.

    Keyed on user_email, resolved to a local id inside each region. See
    resolve_local_user_id() for why passing one numeric id to every region is a
    two-incident bug.

    1. Create erasure request in all regions
    2. Delete from every region, resolving the local id per region
    3. INDEPENDENTLY verify no trace remains anywhere
    4. Only then mark the request completed
    """
    results = {}

    print(f"\n{'='*60}")
    print(f"🔒 GDPR ARTICLE 17 — FULL ERASURE PROCESS")
    print(f"   Data subject: {user_email}")
    print(f"   Reason: {reason}")
    print(f"{'='*60}")

    # Step 1: Create erasure requests in all regions
    request_ids = {}
    for region in ALL_REGIONS:
        local_id = resolve_local_user_id(user_email, region)
        conn = get_connection(region)
        cur = conn.cursor()
        cur.execute("""
            INSERT INTO erasure_requests (user_id, user_email, reason, status)
            VALUES (%s, %s, %s, 'processing')
            RETURNING id
        """, (local_id if local_id is not None else -1, user_email, reason))
        request_ids[region] = cur.fetchone()[0]
        conn.commit()
        conn.close()
        print(f"\n📥 Erasure request #{request_ids[region]} created in {region} "
              f"(local id here: {local_id})")

    # Step 2: Execute deletion in each region
    for region in ALL_REGIONS:
        report = execute_erasure(user_email, request_ids[region], region)
        results[region] = report

    # Step 3: Verify independently, before claiming anything
    residue = verify_erased_everywhere(user_email)
    print(f"\n🔎 Independent post-erasure sweep across {len(ALL_REGIONS)} region(s):")
    if residue:
        for region, tables in residue.items():
            print(f"   ❌ {region}: still holds rows in {tables}")
    else:
        print("   ✅ no trace found in any region")

    # Step 3: Generate summary
    print(f"\n{'='*60}")
    print("📊 ERASURE SUMMARY")
    print(f"{'='*60}")
    for region, report in results.items():
        print(f"\n  {region.upper()}:")
        for table, count, action in report:
            status = "✅" if action == "deleted" else "⚠️"
            print(f"    {status} {table}: {count} records {action}")

    assert not residue, (
        f"erasure claimed success but data remains: {residue}. Never report an "
        f"Article 17 request as completed on the strength of the delete routine's "
        f"own return value."
    )

    print(f"\n✅ Live-database erasure verified for {user_email}")
    print(f"   Rows removed from {len(ALL_REGIONS)} region(s), verified by re-query")
    print(f"   Request trail preserved in erasure_requests")
    print(f"\n⚠️  Still outstanding, and NOT covered by the above:")
    print(f"   • backups and PITR snapshots (next section)")
    print(f"   • caches, search indexes, analytics/warehouse copies, log lines")
    print(f"   • processors and third parties (Article 19 notification)")
    print(f"   'Deleted from the database' is not 'erased'.")
    return results


# ── First: finish the half-done erasure of Anna from the previous section ──
# Note we re-run the FULL process rather than patching eu-north by hand. The
# replica needs its own request record too — "we erased her, trust us" is not
# evidence, and Notebook 4's audit checks for a request in every region.
print("↩️  Completing the erasure we deliberately left half-finished:")
full_gdpr_erasure(ANNA, reason="Completing Art. 17 request — replica was missed")

# ── Then: a clean end-to-end run for a second data subject ──
MARC = "marc.dupont@example.be"
full_gdpr_erasure(
    user_email=MARC,
    reason="User clicked 'Delete My Account' in account settings"
)

## 3b. The copy you cannot DELETE from: backups

The replica was the easy one — it is a live database, so you can run SQL against
it. Backups are the case that actually breaks people's mental model.

A backup is a **point-in-time image**. Last Tuesday's snapshot contains the world
as it was last Tuesday, including the person who asked to be erased on Friday.
And you cannot fix that by editing the snapshot:

- Restores are usually **physical** (file/block level). There is no row to delete.
- Many backups are **immutable by design** — WORM/legal-hold policies exist
  specifically to stop anyone modifying them, including you.
- Rewriting a backup destroys the property that makes it a backup. A snapshot you
  have edited is no longer a faithful image of any moment that ever existed, and
  you have quietly broken your own restore integrity guarantees.

### What regulators actually expect

The established position (see the UK ICO's guidance on the right to erasure, and
equivalent guidance from EU DPAs) is roughly:

1. You may **put the backup beyond use** rather than surgically edit it: the data
   stays only in the backup, is not used for any live purpose, is protected by
   appropriate security, and will be **deleted when the backup ages out** of its
   retention window.
2. You must **tell the data subject** that this is what is happening — that live
   systems are erased and the backup copy will expire on a defined schedule.
3. You must guarantee the data **never comes back to life**. That means a
   **suppression list** (sometimes "do-not-restore" or "tombstone" list): a record
   of erased subjects, consulted at restore time, so a restore does not silently
   resurrect them.
4. Which means your retention window is now a compliance parameter. A backup you
   keep for seven years is a seven-year erasure delay you must be able to defend.

Point 3 is the interesting one for us, because it is a real piece of engineering
and it contains a nasty little paradox: **to reliably forget someone, you must
permanently remember that you forgot them.** Let's build it.


In [ ]:
# ── Backups vs erasure ──────────────────────────────────────────────
# We model a nightly snapshot as a table copy. Real snapshots are physical
# images, but the property that matters here is identical: it is a frozen
# picture of the past that you cannot selectively edit.

import hashlib, hmac, os

BACKUP_USER = "backup.demo@example.nl"

def take_backup(region):
    """Simulates last night's snapshot of the users table."""
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS backup_users_snapshot")
    cur.execute("CREATE TABLE backup_users_snapshot AS SELECT * FROM users")
    cur.execute("SELECT COUNT(*) FROM backup_users_snapshot")
    n = cur.fetchone()[0]
    conn.commit(); conn.close()
    return n

def backup_contains(email, region):
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("SELECT full_name, phone FROM backup_users_snapshot WHERE email = %s", (email,))
    row = cur.fetchone()
    conn.close()
    return row

# 1. A user exists and gets backed up.
for region in ALL_REGIONS:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("DELETE FROM users WHERE email = %s", (BACKUP_USER,))
    cur.execute("""
        INSERT INTO users (email, full_name, phone, country_code, home_region, consent_given)
        VALUES (%s, 'Backup Bram', '+31-6-0000-1111', 'NL', 'eu-west', TRUE)
    """, (BACKUP_USER,))
    conn.commit(); conn.close()

print("💾 BACKUP vs ERASURE")
print("=" * 62)
for region in ALL_REGIONS:
    n = take_backup(region)
    print(f"   🌙 Nightly snapshot taken in {region}: {n} user rows")

# 2. The user exercises Article 17. We do a complete, verified live erasure.
print("\n📮 Data subject requests erasure...")
full_gdpr_erasure(BACKUP_USER, reason="Article 17 request — backup demo")

# 3. Live databases are clean. Now look in the snapshot.
print("\n🔦 Live databases are verified clean. Looking inside the backup:")
still_in_backup = {}
for region in ALL_REGIONS:
    row = backup_contains(BACKUP_USER, region)
    still_in_backup[region] = row
    if row:
        print(f"   ❌ {region} snapshot STILL HOLDS: {row[0]}, phone {row[1]}")

assert all(still_in_backup.values()), (
    "the backup should still contain the erased user — that residue is the "
    "entire point of this section. If the snapshot is clean, the demo is not "
    "modelling a point-in-time backup any more."
)

print("\n   No amount of DELETE against the live database changes this.")
print("   The snapshot is a picture of Tuesday, and on Tuesday he existed.")


In [ ]:
# ── The suppression list: remembering that we forgot ────────────────
# A restore must not resurrect an erased data subject. So we keep a permanent,
# minimal record of WHO was erased — and consult it on every restore.
#
# We store a keyed hash (HMAC-SHA256) of the identifier rather than the
# identifier itself. That way the list can:
#   • answer "was this specific person erased?" for anyone presenting the email
#   • NOT function as a browsable directory of everyone who ever left
# It is still personal data (we can match it to a person, so Recital 26 applies),
# but it is minimised to the single purpose that justifies keeping it.
#
# The key must be managed as a secret and must NOT be stored next to the list —
# an HMAC whose key sits in the same database is a hash with extra steps.
SUPPRESSION_KEY = os.environ.get("SUPPRESSION_KEY", "demo-key-do-not-use-in-prod").encode()

def subject_digest(email):
    return hmac.new(SUPPRESSION_KEY, email.lower().encode(), hashlib.sha256).hexdigest()

def add_to_suppression_list(email, region):
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS erasure_suppression_list (
            subject_digest CHAR(64) PRIMARY KEY,
            erased_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    cur.execute("""
        INSERT INTO erasure_suppression_list (subject_digest) VALUES (%s)
        ON CONFLICT (subject_digest) DO NOTHING
    """, (subject_digest(email),))
    conn.commit(); conn.close()

def restore_from_backup(region, apply_suppression):
    """
    Restores users from the snapshot.

    apply_suppression=False models the naive restore that every DR runbook
    contains by default: bring the data back exactly as it was.

    The digest comparison happens in Python rather than in SQL because the
    restore tool is what holds the HMAC key. Putting the key in the database
    next to the digests would reduce the HMAC to a plain hash.
    """
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("""
        SELECT email, full_name, phone, date_of_birth, country_code,
               home_region, consent_given, consent_date
        FROM backup_users_snapshot
    """)
    rows = cur.fetchall()
    suppressed = set()
    if apply_suppression:
        cur.execute("SELECT subject_digest FROM erasure_suppression_list")
        suppressed = {r[0] for r in cur.fetchall()}
    restored = 0
    for row in rows:
        if apply_suppression and subject_digest(row[0]) in suppressed:
            continue
        cur.execute("""
            INSERT INTO users (email, full_name, phone, date_of_birth, country_code,
                               home_region, consent_given, consent_date)
            VALUES (%s,%s,%s,%s,%s,%s,%s,%s)
            ON CONFLICT (email) DO NOTHING
        """, row)
        restored += cur.rowcount
    conn.commit(); conn.close()
    return restored


print("🔁 RESTORE DRILL")
print("=" * 62)

# ── Drill 1: the naive restore ──
print("\n1️⃣  Restoring from backup with NO suppression list (the default runbook):")
for region in ALL_REGIONS:
    restore_from_backup(region, apply_suppression=False)
resurrected = {r: resolve_local_user_id(BACKUP_USER, r) for r in ALL_REGIONS}
print(f"   Backup Bram after restore: {resurrected}")
assert all(v is not None for v in resurrected.values()), (
    "the naive restore should have resurrected the erased user — if it didn't, "
    "this drill is not demonstrating the resurrection hazard"
)
print("   ❌ He is back. Fully. Name, phone, date of birth.")
print("      Your erasure_requests table still says 'completed'. Your DR runbook")
print("      just silently reversed an Article 17 decision, and nothing anywhere")
print("      logged that it happened.")

# ── Drill 2: erase again, register suppression, restore again ──
print("\n2️⃣  Re-erasing, this time recording him on the suppression list:")
full_gdpr_erasure(BACKUP_USER, reason="Article 17 request — re-erasure after restore")
for region in ALL_REGIONS:
    add_to_suppression_list(BACKUP_USER, region)
print(f"   Suppression digest recorded: {subject_digest(BACKUP_USER)[:16]}...")

print("\n3️⃣  Restoring from the SAME backup, suppression list applied:")
for region in ALL_REGIONS:
    restore_from_backup(region, apply_suppression=True)
after = {r: resolve_local_user_id(BACKUP_USER, r) for r in ALL_REGIONS}
print(f"   Backup Bram after suppressed restore: {after}")
assert all(v is None for v in after.values()), (
    "the suppression list failed to keep the erased subject out of the restore — "
    "this is the control the whole section exists to demonstrate"
)
print("   ✅ Still erased. Everyone else came back.")

print("\n💡 The paradox worth sitting with:")
print("   To forget someone reliably, you must permanently remember that you")
print("   forgot them. The suppression list is itself personal data — it is")
print("   pseudonymised, minimised to one field and one purpose, and justified")
print("   by the erasure obligation itself. Get that reasoning written down")
print("   before an auditor asks why you kept a list of everyone who left.")

# ── Clean up the simulated backup machinery ──
for region in ALL_REGIONS:
    conn = get_connection(region)
    cur = conn.cursor()
    cur.execute("DROP TABLE IF EXISTS backup_users_snapshot")
    cur.execute("DROP TABLE IF EXISTS erasure_suppression_list")
    cur.execute("DELETE FROM users WHERE email = %s", (BACKUP_USER,))
    conn.commit(); conn.close()
print("\n🧹 Backup demo tables dropped.")


## 4. Pseudonymisation is not anonymisation

This section used to be called "The Anonymization Pattern" and it was wrong in
the way that almost every engineering blog post on this topic is wrong. The two
words are not synonyms, they are not points on a spectrum you can hand-wave
between, and GDPR treats them completely differently.

### The legal distinction, precisely

**Pseudonymisation** — Article 4(5): processing personal data so it can no longer
be attributed to a specific data subject *without the use of additional
information*, where that additional information is kept separately and subject to
safeguards.

> **Pseudonymised data is still personal data.** Recital 26: *"Personal data which
> have undergone pseudonymisation, which could be attributed to a natural person
> by the use of additional information should be considered to be information on
> an identifiable natural person."*

So pseudonymisation is a **security measure** — GDPR explicitly rewards it in
Articles 25 and 32 — but it does not take data out of scope. Every right still
applies: access, rectification, **erasure**, portability, objection.

**Anonymisation** — Recital 26 again: GDPR *"does not therefore concern the
processing of such anonymous information"*. Truly anonymous data is outside the
regulation entirely. The bar is correspondingly brutal: identification must be
impossible **for anyone**, "taking account of all the means reasonably likely to
be used" by the controller *or by another person*, considering cost, time, and
available technology **including future technology**.

Article 29 Working Party Opinion 05/2014 gives the working test — data is
anonymous only if you can answer *no* to all three:

| Risk | Question |
|---|---|
| **Singling out** | Can you isolate the records belonging to one individual? |
| **Linkability** | Can you link two records about the same person, here or in another dataset? |
| **Inference** | Can you deduce an attribute of an individual with significant probability? |

### Grade the code we are about to run against that test

Here is what a typical "anonymise the user" routine does, and where it lands:

| What it does | Verdict |
|---|---|
| `full_name = '[REDACTED]'`, `phone = NULL`, `date_of_birth = NULL` | Direct identifiers removed — good, and necessary |
| `email = 'deleted_user_4@anonymized.local'` | ❌ **the row id is embedded in the replacement**. Trivially reversible. |
| The `users.id` primary key is left untouched | ❌ **singling out**: the record is still one specific person, and everything keyed on that id still points at them |
| `addresses`, `orders`, `consent_log` still carry `user_id` | ❌ **linkability**: the whole graph of their behaviour is intact and joined to the same key |
| `country_code`, `home_region`, `created_at` retained | ❌ **quasi-identifiers**: signup timestamp plus country plus order history is often uniquely identifying on its own |

That is **pseudonymisation**, and not a very strong version of it. Calling it
anonymisation in a runbook, a DPIA, or a conversation with a regulator is a
material misstatement — because it asserts the data has left GDPR's scope when
it plainly has not.

### So what do you actually do?

Be honest about which of three things you are doing:

1. **Erasure** — the data is gone. Use it when nothing requires you to keep it.
2. **De-linking under a retention exemption** — keep the specific field the law
   requires (the invoice amount, the transaction date) and destroy the link to
   the person. That is what `execute_erasure` does with `orders`: `user_id = NULL`.
   Still assess the remainder: an "anonymous" order that says *€5,999 Visual
   Studio Enterprise, Munich, 10 April 2024* may single out exactly one customer.
3. **Pseudonymisation** — a security control you apply to data you are *keeping*
   and still treating as personal data.

True anonymisation is a separate discipline (k-anonymity, l-diversity,
differential privacy, aggregation with suppression thresholds) and it is a
property of the **whole released dataset**, never of a single row. You cannot
anonymise one record in a table, because anonymity is defined by how many other
records that record hides among.

Let's run the routine, then measure how badly it fails the test.


In [ ]:
# ── Pseudonymisation demo (correctly labelled this time) ───────────

def pseudonymise_user(user_id, region, pseudonym):
    """
    Removes direct identifiers from a user's record while keeping the row.

    This is PSEUDONYMISATION (Article 4(5)), not anonymisation. The output is
    still personal data under Recital 26 and every data-subject right still
    applies to it. Use when a retention exemption means you must keep the row
    but do not need the identifiers.

    Note what changed from the naive version:
      • the replacement email no longer embeds the row id
      • addresses are DELETED rather than string-replaced, because a redacted
        address row still says "this person had an address in this country"
      • orders are de-linked (user_id -> NULL), matching execute_erasure()
      • we return an honest report of what remains linkable
    """
    conn = get_connection(region)
    cur = conn.cursor()

    cur.execute("SELECT full_name, email FROM users WHERE id = %s", (user_id,))
    user = cur.fetchone()
    if not user:
        print(f"   User {user_id} not found in {region}")
        conn.close()
        return None

    print(f"\n   Before: {user[0]} ({user[1]})")

    # `pseudonym` is a random token generated ONCE by the caller and applied
    # identically in every region. Two regions must not invent different
    # pseudonyms for the same person: that splits one data subject into two
    # unlinkable records, and the next access or erasure request will only find
    # one of them. The mapping from token back to person is the "additional
    # information" of Article 4(5) — if you need reversibility it lives in a
    # separate, access-controlled store, never derivable from the pseudonym.
    cur.execute("""
        UPDATE users SET
            email          = %s,
            full_name      = '[REDACTED]',
            phone          = NULL,
            date_of_birth  = NULL,
            consent_given  = FALSE,
            consent_date   = NULL,
            updated_at     = NOW()
        WHERE id = %s
    """, (pseudonym, user_id))

    # Delete addresses outright. '[REDACTED]' in a street column still leaves a
    # row asserting "this person lived somewhere in Germany" — a quasi-identifier
    # dressed up as redaction.
    cur.execute("DELETE FROM addresses WHERE user_id = %s", (user_id,))
    addr_deleted = cur.rowcount

    # De-link orders rather than deleting them (statutory retention), same as
    # execute_erasure does.
    cur.execute("UPDATE orders SET user_id = NULL WHERE user_id = %s", (user_id,))
    orders_delinked = cur.rowcount

    cur.execute("DELETE FROM consent_log WHERE user_id = %s", (user_id,))
    consent_deleted = cur.rowcount

    conn.commit()

    cur.execute("SELECT full_name, email, phone, date_of_birth, country_code, created_at FROM users WHERE id = %s", (user_id,))
    anon = cur.fetchone()
    conn.close()

    print(f"   After:  {anon[0]} ({anon[1]})")
    print(f"           phone={anon[2]}, dob={anon[3]}")
    print(f"           addresses deleted: {addr_deleted} | orders de-linked: {orders_delinked} | consent rows deleted: {consent_deleted}")
    return {
        "user_id": user_id,
        "retained_country": anon[4],
        "retained_created_at": anon[5],
        "orders_delinked": orders_delinked,
    }


print("🎭 PSEUDONYMISATION DEMO")
print("=" * 62)
print("Scenario: Hans Müller has a statutory retention hold on his invoices.")
print("We cannot delete the invoices, so we strip identifiers from his record.")

import uuid

HANS = "hans.mueller@example.de"
HANS_PSEUDONYM = f"pseudonymised-{uuid.uuid4().hex[:12]}@invalid.example"

# Apply in EVERY region — same lesson as erasure. Pseudonymising the primary and
# leaving the replica holding the identified original is not pseudonymisation of
# anything; it just means the plaintext lives in one place instead of two.
assert resolve_local_user_id(HANS, "eu-west") is not None, (
    f"{HANS} is not in eu-west — start from fresh containers "
    f"(docker compose down -v && docker compose up -d) before running this notebook"
)

hans_ids = {}
result = None
for _region in ALL_REGIONS:
    _id = resolve_local_user_id(HANS, _region)
    hans_ids[_region] = _id
    if _id is not None:
        print(f"\n── {_region} (local id {_id}) ──")
        _res = pseudonymise_user(_id, _region, HANS_PSEUDONYM)
        if _region == "eu-west":
            result = _res

hans_id = hans_ids["eu-west"]

# The identified original must be gone from every region, and the pseudonym must
# be the SAME everywhere so the person is still one person.
for _region in ALL_REGIONS:
    assert resolve_local_user_id(HANS, _region) is None, (
        f"the identified record still exists in {_region} — pseudonymising only "
        f"the primary leaves the replica holding the plaintext"
    )
    assert resolve_local_user_id(HANS_PSEUDONYM, _region) is not None, (
        f"the pseudonymised record is missing from {_region}"
    )
print(f"\n✅ Same pseudonym applied in all {len(ALL_REGIONS)} regions: {HANS_PSEUDONYM}")

# ── Now grade the result honestly against the WP29 three-part test ──
print("\n" + "=" * 62)
print("📏 IS THIS ANONYMOUS? (Article 29 WP Opinion 05/2014 test)")
print("=" * 62)

conn = get_connection("eu-west")
cur = conn.cursor()

# 1. Singling out — is there still exactly one row that is this person?
cur.execute("SELECT COUNT(*) FROM users WHERE id = %s", (hans_id,))
singled_out = cur.fetchone()[0] == 1

# 2. Linkability — does anything still join to this individual's key?
cur.execute("SELECT COUNT(*) FROM data_residency_log WHERE user_id = %s", (hans_id,))
linkable_rows = cur.fetchone()[0]

# 3. Inference — do the retained quasi-identifiers narrow him down?
cur.execute("""
    SELECT COUNT(*) FROM users
    WHERE country_code = %s AND date_trunc('day', created_at) = date_trunc('day', %s::timestamp)
""", (result["retained_country"], result["retained_created_at"]))
cohort_size = cur.fetchone()[0]
conn.close()

print(f"   Singling out : {'YES' if singled_out else 'no':<4} — row id {hans_id} is still one specific person")
print(f"   Linkability  : {'YES' if linkable_rows else 'no':<4} — {linkable_rows} audit row(s) still key on that id")
print(f"   Inference    : cohort of {cohort_size} — users sharing his country AND signup day")
print(f"                  (k-anonymity of {cohort_size}; k=1 means he is uniquely identified")
print(f"                   by those two 'non-identifying' fields alone)")

assert singled_out, (
    "the pseudonymised row should still be singled out — that is precisely why "
    "this is pseudonymisation and not anonymisation. If the row is gone, this "
    "cell is measuring an erasure, and the lesson is lost."
)

print("\n❌ VERDICT: NOT ANONYMOUS.")
print("   This is pseudonymised personal data. Recital 26 applies in full.")
print("   Hans can still submit an Article 15 access request against this row,")
print("   and an Article 17 erasure request for everything not covered by the")
print("   retention exemption. Nothing about it is 'statistically invisible'.")
print()
print("✅ What you may honestly claim:")
print("   'We removed direct identifiers and de-linked the retained invoices.")
print("    The residual record is pseudonymised personal data, retained under")
print("    Article 17(3)(b) for the statutory accounting period, and deleted")
print("    when that period expires.'")
print()
print("🚫 What you may not claim:")
print("   'The user has been anonymised.'  — that asserts the data left GDPR's")
print("   scope, which would end every one of their rights. It has not.")


## 5. Why Microsoft Builds This Into Azure

### The Scale of the Problem

Microsoft processes erasure requests for:
- **Azure** (cloud infrastructure)
- **Microsoft 365** (email, documents, Teams)
- **LinkedIn** (professional profiles)
- **Xbox** (gaming accounts)
- **GitHub** (developer accounts)

Each service has PII scattered across **dozens of databases**, **multiple regions**, and **various backup systems**.

### Azure's Tooling (correct product names as of 2025)

1. **Microsoft Purview Data Map** — discovers and classifies data estates. (It was called *Azure* Purview until the 2022 rebrand; the "Azure Data Map" in an earlier draft of this notebook was not a product.)
2. **Microsoft Purview Data Lineage** — tracks where data flows.
3. **Microsoft Purview Compliance Manager** — produces a *readiness score* against control frameworks. It scores the controls you have evidence for. It does not, and Microsoft does not claim it does, determine that you are compliant.
4. **Per-service deletion semantics** — note there is no "paired region deletion" feature. Erasure reaching every replica is *your* responsibility, implemented per service.

### Key Insight

`ON DELETE CASCADE` is a good default and it is nowhere near sufficient:

- It only covers rows with a **declared foreign key**. The naive-DELETE demo above showed exactly what happens to the legacy table that has none.
- It stops at the **database boundary**. Caches, search indexes, object storage, message queues, warehouse copies, log aggregators and third-party processors have no idea a cascade happened.
- It does not reach **backups**, as we spent a whole section proving.
- It is **replicated, not coordinated**: a cascade in one region says nothing about the other.

The real primitive is not the constraint — it is the **inventory**. You cannot erase what you have not written down that you hold, which is why Article 30's record of processing activities is a prerequisite for Article 17 rather than a separate chore.

## 6. The Other Side of Erasure: Right to Access (Article 15)

Article 15 and Article 17 are the same engineering problem seen from two sides:
both require you to enumerate every place a person's data lives. If your export
misses a table, your erasure misses it too — the export is a cheap, non-destructive
way to test the completeness of your inventory.

A request under Article 15 is usually called a **Data Subject Access Request (DSAR)**:

> "The user has the right to obtain a copy of the personal data undergoing
> processing, in a **commonly used, machine-readable format**."

Article 20 ("data portability") is narrower than people assume: it applies only
to data the subject *provided*, processed by automated means, on the basis of
**consent or contract** — not to data you inferred about them, and not to
processing under legitimate interests. Article 15 access is the broad right;
Article 20 portability is the machine-readable-transfer right.

So build the export first — the code is almost identical, you just `SELECT`
instead of `DELETE`, and running both against the same table list is how you keep
them honest.


In [ ]:
# ── DSAR: Export all of a user's personal data as JSON ─────
import json

def export_user_data(user_id, region):
    """
    GDPR Article 15 / 20 — return *all* personal data we hold on the user
    in a machine-readable format (JSON). This is what you'd email the user
    when they submit a "download my data" request.
    """
    conn = get_connection(region)
    cur = conn.cursor()

    export = {"region": region, "generated_at": datetime.now().isoformat()}

    # Each query returns columns + rows so the output is self-describing.
    #
    # This table list is the SAME inventory PII_TABLES uses for erasure — and it
    # must stay that way. A table that appears in one list and not the other is
    # either data you export but never erase, or data you erase but never
    # disclose. Both are defects; the assertion below catches drift.
    for table, sql in [
        ("users",              "SELECT * FROM users              WHERE id      = %s"),
        ("addresses",          "SELECT * FROM addresses          WHERE user_id = %s"),
        ("orders",             "SELECT * FROM orders             WHERE user_id = %s"),
        ("consent_log",        "SELECT * FROM consent_log        WHERE user_id = %s"),
        ("data_residency_log", "SELECT * FROM data_residency_log WHERE user_id = %s"),
    ]:
        cur.execute(sql, (user_id,))
        cols = [d[0] for d in cur.description]
        export[table] = [dict(zip(cols, map(str, row))) for row in cur.fetchall()]

    conn.close()
    return export

# Demo: export everything we have on Claire Martin
CLAIRE = "claire.martin@example.fr"
claire_id = resolve_local_user_id(CLAIRE, "eu-west")
data = export_user_data(user_id=claire_id, region="eu-west")
print(f"📤 DSAR export preview for {CLAIRE} (local id {claire_id}):")
print(json.dumps(data, indent=2, default=str)[:1200], "...\n")

# ── Keep the access inventory and the erasure inventory in lockstep ──
exported_tables = {t for t in data if t not in ("region", "generated_at")}
erasure_tables = set(PII_TABLES)
only_erased = erasure_tables - exported_tables
only_exported = exported_tables - erasure_tables - {"data_residency_log"}
assert not only_erased, (
    f"tables {sorted(only_erased)} are erased but never disclosed in a DSAR — "
    f"you are holding data you do not tell the data subject about"
)
assert not only_exported, (
    f"tables {sorted(only_exported)} are disclosed in a DSAR but never erased — "
    f"an Article 17 request against this system would leave them behind"
)
print(f"✅ Access and erasure cover the same tables: {sorted(erasure_tables)}")
print( "   (data_residency_log is exported but deliberately retained — it is the")
print( "    evidence that the erasure happened. Justify that retention explicitly.)")

print("\n💡 In production you would:")
print("   1. Verify the requester's identity — Article 12(6). Handing a stranger")
print("      a complete PII dossier because they typed an email address is the")
print("      classic DSAR-as-attack-vector, and it has happened repeatedly.")
print("   2. Run this against *every* region and merge, resolving the local ids")
print("      per region — same trap as erasure.")
print("   3. Include data you *inferred*, not just data they typed in. Article 15")
print("      covers everything you process about them.")
print("   4. Deliver as a signed, expiring download link, and log the disclosure.")


## 🎯 Key Takeaways

1. **Right to erasure** (Art. 17) is a **conditional** right, not a delete button — it attaches to the grounds in 17(1) and yields to the exemptions in 17(3), *to the extent necessary*.
2. **Naive `DELETE FROM users`** leaves orphan PII. `ON DELETE CASCADE` helps and is not enough: it only follows declared foreign keys, and it stops at the database boundary.
3. **Never key a cross-region operation on a local `SERIAL` id.** The sequences are independent. The bug erases an innocent third party in one region while leaving the requester intact in the other — silently, and symmetrically.
4. **Verify erasure independently.** A delete routine that reports its own success will report success when it deleted nothing. We watched `erasure_requests.status = 'completed'` be a lie while the replica still held everything.
5. **Backups cannot be edited, and you should not want them to be.** The accepted answer is: put the backup beyond use, let it expire on a defined retention schedule, tell the data subject that is what is happening, and maintain a **suppression list** so a restore never resurrects them.
6. **To forget someone reliably you must permanently remember that you forgot them.** The suppression list is itself pseudonymised personal data; minimise it and write down why you keep it.
7. **Pseudonymisation ≠ anonymisation.** Pseudonymised data is still personal data (Recital 26) with every right attached. Anonymous data is outside GDPR entirely, and the bar — no singling out, no linkability, no inference, "by anyone", "including future technology" — is much higher than nulling a few columns. We measured our own output and it failed all three.
8. **Anonymity is a property of a dataset, not of a row.** You cannot anonymise one record, because anonymity is defined by how many other records it hides among.
9. **The legal clock is one month** (Art. 12(3)), extendable by two further months for complex requests if you notify within the first month. "30 days" is a workable internal SLA, not the rule.
10. **Article 15 access and Article 17 erasure share one inventory.** Keep the table lists in lockstep — we assert on it — because a table missing from one is missing from both.

## ⏭️ Next Up

In **Notebook 4**, we'll build the audit that has to catch everything above — including the half-finished erasure, which the original version of that audit reported as a clean bill of health.
